In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from dotenv import load_dotenv
import os

In [2]:
load_dotenv()

True

In [3]:
access_key = os.getenv("Access_key_ID")
secret_key = os.getenv("Secret_access_key")
bucket = os.getenv("BUCKET_NAME")
region = os.getenv("REGION_NAME")

In [4]:
spark = SparkSession.builder \
    .appName("S3DataTransformation") \
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.1") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.access.key", access_key) \
    .config("spark.hadoop.fs.s3a.secret.key", secret_key) \
    .config("spark.hadoop.fs.s3a.endpoint", "s3.amazonaws.com") \
    .config("spark.hadoop.fs.s3a.region", region) \
    .getOrCreate()

25/04/17 09:45:50 WARN Utils: Your hostname, brempong-HP-EliteBook-840-G7-Notebook-PC resolves to a loopback address: 127.0.1.1; using 192.168.36.43 instead (on interface wlp0s20f3)
25/04/17 09:45:50 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/brempong/Ecommerce-DataLakehouse/venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/brempong/.ivy2/cache
The jars for the packages stored in: /home/brempong/.ivy2/jars
org.apache.hadoop#hadoop-aws added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-73255a2d-7b16-4d34-a68d-02fd324ae5b6;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.3.1 in central
	found com.amazonaws#aws-java-sdk-bundle;1.11.901 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
:: resolution report :: resolve 279ms :: artifacts dl 13ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.11.901 from central in [default]
	org.apache.hadoop#hadoop-aws;3.3.1 from central in [default]
	org.wildfly.openssl#wildfly-openssl;1.0.7.Final from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	----------------------------

In [5]:
spark

In [7]:
orders_folder_path = f"s3a://{bucket}/raw-data/orders_apr_2025/"
orders = spark.read.csv(orders_folder_path, header=True, inferSchema=True)
orders.show()

25/04/17 09:48:14 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


+---------+--------+-------+-------------------+------------+----------+
|order_num|order_id|user_id|    order_timestamp|total_amount|      date|
+---------+--------+-------+-------------------+------------+----------+
|       11|   13000|   6260|2025-04-07 10:44:00|      444.48|2025-04-07|
|       40|   13001|   4436|2025-04-07 16:59:00|      231.16|2025-04-07|
|       44|   13002|   1271|2025-04-07 21:29:00|      245.62|2025-04-07|
|       43|   13003|   9974|2025-04-07 03:01:00|      147.55|2025-04-07|
|       54|   13004|   7561|2025-04-07 11:09:00|      137.04|2025-04-07|
|       35|   13005|   8784|2025-04-07 01:02:00|      150.17|2025-04-07|
|       80|   13006|   8694|2025-04-07 05:20:00|      162.56|2025-04-07|
|       95|   13007|   6024|2025-04-07 01:36:00|      237.47|2025-04-07|
|       85|   13008|   5508|2025-04-07 19:36:00|      110.47|2025-04-07|
|       71|   13009|   9916|2025-04-07 16:55:00|      146.99|2025-04-07|
|       31|   13010|   6641|2025-04-07 09:39:00|   

In [9]:
order_items_folder_path = f"s3a://{bucket}/raw-data/order_items_apr_2025/"
order_items = spark.read.csv(order_items_folder_path, header=True, inferSchema=True)
order_items.show()

+-----+--------+-------+----------------------+----------+-----------------+---------+-------------------+----------+
|   id|order_id|user_id|days_since_prior_order|product_id|add_to_cart_order|reordered|    order_timestamp|      date|
+-----+--------+-------+----------------------+----------+-----------------+---------+-------------------+----------+
|10912|   12000|   6024|                    16|       380|                1|        0|2025-04-05 04:37:00|2025-04-05|
|10913|   12000|   6024|                    15|       469|                2|        0|2025-04-05 04:37:00|2025-04-05|
|10914|   12000|   6024|                    21|       205|                3|        0|2025-04-05 04:37:00|2025-04-05|
|10915|   12000|   6024|                    27|       284|                4|        1|2025-04-05 04:37:00|2025-04-05|
|10916|   12001|   2459|                    22|       184|                1|        1|2025-04-05 13:00:00|2025-04-05|
|10917|   12001|   2459|                    16|       76

In [11]:
product_folder_path = f"s3a://{bucket}/raw-data/products/"
product = spark.read.csv(product_folder_path, header=True, inferSchema=True)
product.show()

+----------+-------------+-----------+-------------------+
|product_id|department_id| department|       product_name|
+----------+-------------+-----------+-------------------+
|         1|            4|      Books|    Product_1_Store|
|         2|            2|      Books|    Product_2_There|
|         3|            4|      Books|     Product_3_Hand|
|         4|            6|     Sports|       Product_4_Tv|
|         5|            1|       Toys|     Product_5_Easy|
|         6|            1|     Sports|    Product_6_Woman|
|         7|            5|       Home|     Product_7_Yard|
|         8|            5|      Books|  Product_8_Manager|
|         9|            6|       Toys|     Product_9_Face|
|        10|            4|     Sports|   Product_10_Sport|
|        11|            2|   Clothing|  Product_11_Parent|
|        12|            2|      Books| Product_12_Contain|
|        13|            4|     Sports|Product_13_Audience|
|        14|            1|       Home|     Product_14_Jo

In [14]:
# Join order_products_df with orders_df
joined_df = order_items.alias("op") \
    .join(orders.alias("o"), on="order_id", how="inner") \
    .join(product.alias("p"), on="product_id", how="inner")

In [15]:
joined_df.show()

+----------+--------+-----+-------+----------------------+-----------------+---------+-------------------+----------+---------+-------+-------------------+------------+----------+-------------+----------+--------------------+
|product_id|order_id|   id|user_id|days_since_prior_order|add_to_cart_order|reordered|    order_timestamp|      date|order_num|user_id|    order_timestamp|total_amount|      date|department_id|department|        product_name|
+----------+--------+-----+-------+----------------------+-----------------+---------+-------------------+----------+---------+-------+-------------------+------------+----------+-------------+----------+--------------------+
|       380|   12000|10912|   6024|                    16|                1|        0|2025-04-05 04:37:00|2025-04-05|       40|   6024|2025-04-05 04:37:00|      242.81|2025-04-05|            4|     Books|    Product_380_View|
|       469|   12000|10913|   6024|                    15|                2|        0|2025-04-05

In [17]:
final_df = joined_df.select(
    "op.id",
    "op.order_id",
    "op.user_id",
    "op.days_since_prior_order",
    "op.product_id",
    "p.product_name",
    "p.department_id",
    "p.department",
    "op.add_to_cart_order",
    "op.reordered",
    "o.order_num",
    "o.total_amount",
    "op.order_timestamp",
    "op.date"
)

# Show the final result
final_df.show()

+-----+--------+-------+----------------------+----------+--------------------+-------------+----------+-----------------+---------+---------+------------+-------------------+----------+
|   id|order_id|user_id|days_since_prior_order|product_id|        product_name|department_id|department|add_to_cart_order|reordered|order_num|total_amount|    order_timestamp|      date|
+-----+--------+-------+----------------------+----------+--------------------+-------------+----------+-----------------+---------+---------+------------+-------------------+----------+
|10912|   12000|   6024|                    16|       380|    Product_380_View|            4|     Books|                1|        0|       40|      242.81|2025-04-05 04:37:00|2025-04-05|
|10913|   12000|   6024|                    15|       469|Product_469_Execu...|            1|  Clothing|                2|        0|       40|      242.81|2025-04-05 04:37:00|2025-04-05|
|10914|   12000|   6024|                    21|       205| Produc

In [18]:
final_df = final_df.withColumn("order_time", date_format(col("order_timestamp"), "HH:mm:ss"))
final_df.show()

+-----+--------+-------+----------------------+----------+--------------------+-------------+----------+-----------------+---------+---------+------------+-------------------+----------+----------+
|   id|order_id|user_id|days_since_prior_order|product_id|        product_name|department_id|department|add_to_cart_order|reordered|order_num|total_amount|    order_timestamp|      date|order_time|
+-----+--------+-------+----------------------+----------+--------------------+-------------+----------+-----------------+---------+---------+------------+-------------------+----------+----------+
|10912|   12000|   6024|                    16|       380|    Product_380_View|            4|     Books|                1|        0|       40|      242.81|2025-04-05 04:37:00|2025-04-05|  04:37:00|
|10913|   12000|   6024|                    15|       469|Product_469_Execu...|            1|  Clothing|                2|        0|       40|      242.81|2025-04-05 04:37:00|2025-04-05|  04:37:00|
|10914|   

In [20]:
final_df = final_df.withColumn("reordered", when(col("reordered") == 1, "Reorder").otherwise("Not_Reorder"))
final_df.show()


+-----+--------+-------+----------------------+----------+--------------------+-------------+----------+-----------------+-----------+---------+------------+-------------------+----------+----------+
|   id|order_id|user_id|days_since_prior_order|product_id|        product_name|department_id|department|add_to_cart_order|  reordered|order_num|total_amount|    order_timestamp|      date|order_time|
+-----+--------+-------+----------------------+----------+--------------------+-------------+----------+-----------------+-----------+---------+------------+-------------------+----------+----------+
|10912|   12000|   6024|                    16|       380|    Product_380_View|            4|     Books|                1|Not_Reorder|       40|      242.81|2025-04-05 04:37:00|2025-04-05|  04:37:00|
|10913|   12000|   6024|                    15|       469|Product_469_Execu...|            1|  Clothing|                2|Not_Reorder|       40|      242.81|2025-04-05 04:37:00|2025-04-05|  04:37:00|


In [31]:
final_df = final_df.repartition(15, col("date")).sortWithinPartitions(col("department"), col("reordered"))
final_df.show()

+-----+--------+-------+----------------------+----------+--------------------+-------------+----------+-----------------+-----------+---------+------------+-------------------+----------+----------+
|   id|order_id|user_id|days_since_prior_order|product_id|        product_name|department_id|department|add_to_cart_order|  reordered|order_num|total_amount|    order_timestamp|      date|order_time|
+-----+--------+-------+----------------------+----------+--------------------+-------------+----------+-----------------+-----------+---------+------------+-------------------+----------+----------+
|16376|   13001|   4436|                    26|       194|      Product_194_Us|            3|     Books|                7|Not_Reorder|       40|      231.16|2025-04-07 16:59:00|2025-04-07|  16:59:00|
|16377|   13002|   1271|                    24|       547| Product_547_Morning|            4|     Books|                1|Not_Reorder|       44|      245.62|2025-04-07 21:29:00|2025-04-07|  21:29:00|


In [19]:
df.select("date").distinct().show()

+----------+
|      date|
+----------+
|2025-04-05|
|2025-04-14|
|2025-04-07|
|2025-04-08|
|2025-04-11|
|2025-04-09|
|2025-04-13|
|2025-04-01|
|2025-04-02|
|2025-04-03|
|2025-04-12|
|2025-04-15|
|2025-04-04|
|2025-04-06|
|2025-04-10|
+----------+



In [26]:
# Step 1: Users Table
users_df = final_df.select("user_id").dropDuplicates()

# Step 2: Departments Table
departments_df = final_df.select("department_id", "department").dropDuplicates()

# Step 3: Products Table (normalize product to include department as FK)
products_df = final_df.select("product_id", "product_name", "department_id").dropDuplicates()

# Step 4: Orders Table
orders_df = final_df.select(
    "order_id",
    "user_id",
    "days_since_prior_order",
    "order_num",
    "total_amount",
    "date",
    "order_time"
).dropDuplicates(["order_id"])

# Step 5: Order Items Table (fact table)
order_items_df = final_df.select(
    "id",
    "order_id",
    "product_id",
    "add_to_cart_order",
    "reordered"
)


In [27]:
users_df.show()

+-------+
|user_id|
+-------+
|   2366|
|   8592|
|   4935|
|   2387|
|   3698|
|   6623|
|   6773|
|   1127|
|   4190|
|   2393|
|   1699|
|   8222|
|   5173|
|   5117|
|   7066|
|   4239|
|   6597|
|   5223|
|   2721|
|   4684|
+-------+
only showing top 20 rows



In [28]:
orders_df.show()

+--------+-------+----------------------+---------+------------+----------+----------+
|order_id|user_id|days_since_prior_order|order_num|total_amount|      date|order_time|
+--------+-------+----------------------+---------+------------+----------+----------+
|   10000|   1990|                    10|       90|      229.53|2025-04-01|  11:27:00|
|   10001|   5057|                     4|       41|      131.93|2025-04-01|  17:53:00|
|   10002|   7864|                    23|       22|       251.9|2025-04-01|  02:26:00|
|   10003|   3131|                    30|       99|      487.49|2025-04-01|  01:24:00|
|   10004|   9621|                     2|       51|      365.46|2025-04-01|  11:48:00|
|   10005|   8777|                    12|       86|       84.22|2025-04-01|  18:43:00|
|   10006|   2135|                    14|       42|       380.7|2025-04-01|  21:33:00|
|   10007|   7672|                     8|       30|      464.95|2025-04-01|  04:10:00|
|   10008|   1941|                     8|  

In [29]:
order_items_df.show()

+-----+--------+----------+-----------------+-----------+
|   id|order_id|product_id|add_to_cart_order|  reordered|
+-----+--------+----------+-----------------+-----------+
|10912|   12000|       380|                1|Not_Reorder|
|10913|   12000|       469|                2|Not_Reorder|
|10914|   12000|       205|                3|Not_Reorder|
|10915|   12000|       284|                4|    Reorder|
|10916|   12001|       184|                1|    Reorder|
|10917|   12001|       762|                2|    Reorder|
|10918|   12001|       107|                3|    Reorder|
|10919|   12002|       207|                1|    Reorder|
|10920|   12002|       572|                2|    Reorder|
|10921|   12002|       669|                3|    Reorder|
|10922|   12002|       667|                4|    Reorder|
|10923|   12002|       685|                5|    Reorder|
|10924|   12002|       720|                6|Not_Reorder|
|10925|   12002|       630|                7|    Reorder|
|10926|   1200